
**Watson Event Archiving Tool** | Author: Michael Rostom

### Setup
- Get a Gemini API key, which you can get for free from Google AI studio. See [Online Guide](https://www.stephenwthomas.com/azure-integration-thoughts/how-to-get-free-google-gemini-api-access-step-by-step-guide-for-2025/) for help.

- Press the 🗝 Secrets tab on the left sidebar, add the API key with name: GEMINI_API_KEY and paste the API key into the value field. Make Sure that Notebook access is enabled.

This is how it should look like at the end of this section:

![Screenshot of secrets tab](https://i.imgur.com/7w6nYuZ.png)

To Use:
- Press ▶ Run all at the top menu
- It will take a couple minutes to set things up (~2-3 minutes)
- Scroll all the way to the bottom, and input the link.



In [ ]:
%%capture
!pip install google-colab-selenium

In [ ]:
import google_colab_selenium as gs
from google.genai import Client
from selenium import webdriver
from google.colab import userdata
from IPython.display import HTML
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from typing import List

month_dict = {
    "january": "1",
    "february": "2",
    "march": "3",
    "april": "4",
    "may": "5",
    "june": "6",
    "july": "7",
    "august": '8',
    "september": '9',
    "october": '10',
    "november": '11',
    "december": '12'}
def convert_month_to_number(month: str) -> str:
  try:
    res: str = month_dict[month]
    return res
  except KeyError:
    return ""

def open_url(url: str, driver: webdriver.Chrome) -> None:
  driver.get(url)
  WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#lw_cal_event_detail h1")))

def call_gemini(client: Client, prompt: str, context: str) -> str:
  response = client.models.generate_content(
      model="gemini-flash-latest",
      contents= "Execute the following command: " + prompt +
      "\n" +
      "Given the following context:" + context,
      config={
        "thinking_config": {"thinking_budget": 0}
    })
  if response.text is None:
    raise Exception("Gemini didn't return an answer")
  return response.text

def call_gemini_lite(client: Client, prompt: str, context: str) -> str:
  response = client.models.generate_content(
      model="gemini-flash-lite-latest",
      contents= "Execute the following command: " + prompt +
      "\n" +
      "Given the following context:" + context)
  if response.text is None:
    raise Exception("Gemini didn't return an answer")
  return response.text

def validate_gemini_res(res: str, expected: List[str], topic:str) -> str:
  if res in expected:
    return res
  else:
    print("Gemini Failed to pick from standardize list for "+ topic)
    return ""

In [ ]:
from typing import List
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from os import getenv
from google import genai
from google.colab import data_table
import pandas as pd
import re

# Setup Gemini
api_key = userdata.get('GEMINI_API_KEY')
gemini_client = genai.Client(api_key=api_key)


def extract_event_details(url: str) -> dict:
  print("Opening URL...")
  open_url(url, driver)

  # Collect event details
  print("Extracting event details...")
  event_details = {}

  # Title is retrieved from the tab name
  event_details["event_name"] = driver.title.split('|')[0].strip()

  try:
    # Date is captured from the web page then converted to "MM/DD/YYYY" format
    numeric_date_list: List[str] = driver.find_element(By.ID,
                                                      "lw_cal_this_day").text.split(
      " ")
    month_num: str = convert_month_to_number(numeric_date_list[0].lower())
    day_num: str = numeric_date_list[1][:-1]
    year_num: str = numeric_date_list[2]
    event_details["date"] = month_num + "/" + day_num + "/" + year_num

    # Semester is calculated from the month
    month_num_int: int = int(month_num)
    if 8 <= month_num_int <= 12:
      event_details["semester"] = "FALL"
    elif 1 <= month_num_int <= 5:
      event_details["semester"] = "SPRING"
    elif 6 <= month_num_int <= 8:
      event_details["semester"] = "SUMMER"
    else:
      raise ValueError("Invalid month num")

    # Year is inputted directly
    event_details["year"] = year_num

  except NoSuchElementException:
    event_details["date"] = ""
    event_details["semester"] = ""
    event_details["year"] = ""

  if event_details["date"] == "":
    print("Failed to extract date")
  if event_details["semester"] == "":
    print("Failed to extract semester")
  if event_details["year"] == "":
    print("Failed to extract year")

  try:
    # Description is extracted from the web page
    event_description: str = driver.find_element(By.CLASS_NAME,
                                                      "lw_calendar_event_description").text
  except NoSuchElementException:
    # Had to exit here because all other fields depend on description
    raise Exception("Description not found. Can't continue. Exiting..")

  # Keywords are extracted by gemini from description
  event_details["keywords"] = call_gemini_lite(gemini_client,
                                          "Extract keywords from the event description\n"
                                          "Only return the keywords separated by commas\n",
                                          event_description)

  # Topics are extracted by gemini from the description
  event_details["topics"] = call_gemini_lite(gemini_client,
                                        "Extract key themes from the event description\n"
                                        "Only return the one to three word themes separated by commas\n",
                                        event_description)

    # Research Theme is chosen by gemini from the description
  possible_themes = ["Research", "Security", "Development", "Governance"]
  theme_res = call_gemini(gemini_client,
                                                "Choose from: Security, Development, Governance (definitions provided below)\n"
                                                "Only Return the chosen theme name\n"
                                                "Research Theme Definitions\n"
                                                "Security: Covers traditional and emerging global security concerns, including climate change, pandemics, cyber threats, and post-conflict reconstruction.\n"
                                                "Development: Focuses on inequality, governance, urban transformation, democracy, and global economic systems.\n"
                                                "Governance: Explores how globalization affects political and economic institutions and the need for new forms of global governance.\n",
                                                event_description)
    ## Validate Gemini Results
  event_details["research_theme"] = validate_gemini_res(theme_res, possible_themes, "research_theme")

    # Link is inputted directly
  event_details["link"] = url

  # Professor is empty for now
  event_details["professor"] = ""

  # The right column is retrieved from the web page
  # This column includes data for location, room, sponsor
  try:
    right_col: List[str] = driver.find_element(By.ID,
                                              "lw_cal_event_detail_cols_right").text.split(
      '\n')
    # Start with empty here to ensure ordering of columns in the cells
    event_details["center"] = ""
    event_details["Location"] = ""
    event_details["Building"] = ""
    sponsor_found:bool = False
    location_found:bool = False
    building_found:bool = False

    for line in right_col:
      if "Sponsor" in line:
        # Sponsor is the same as center
        event_details["center"] = line.split(':')[1]
        sponsor_found = True
      elif "Room" in line:
        event_details["Location"] = line.split(':')[1]
        location_found = True
      elif "Location" in line:
        event_details["Building"] = line.split(':')[1]
        building_found = True
  except:
    print("Failed to extract right column")
  if not sponsor_found:
    print("Failed to extract center")
  if not location_found:
    print("Failed to extract Location")
  if not building_found:
    print("Failed to extract Building")

  # Public is yes, manually changed to no if needed
  event_details["public"] = "Y"

  # Youtube Link is extracted by gemini from the description
  try:
    event_details["yt_link"] = driver.find_element(By.XPATH,
                                                 "/html/body/div[1]/main/div/div[1]/section/div[2]/div[2]/div/div/div/div[1]/a").get_attribute("href")

  except NoSuchElementException:
    event_details["yt_link"] = ""
    print("Failed to extract yt_link")


  # Talent is extracted by gemini from the description
  talent_num_max = 5
  talent_list: List[str] = call_gemini_lite(gemini_client,
                                       "Extract the talent of the event\n"
                                       "Only names of people actually attending or speaking at the event\n"
                                       "return separated by commas\n",
                                       event_description).split(",")
  talent_num = len(talent_list)
  if talent_num > talent_num_max:
    print("Too many talents, only entering the first " + str(talent_num_max))

  for i in range(talent_num_max):
    if i < talent_num:
      event_details["talent" + str(i + 1)] = talent_list[i]
    else:
      event_details["talent" + str(i + 1)] = ""



  # Country is extracted by gemini from the description
  event_details["country"] = call_gemini_lite(gemini_client,
                                        "Extract the country of the event\n"
                                        "Only return the country name\n",
                                        event_description)


  # Region is extracted by Gemini from the description
  possible_regions = ["Africa",
                      "Brazil",
                      "China",
                      "Europe",
                      "India & South Asia",
                      "Latin America & Caribbean",
                      "Middle East",
                      "Russia",
                      "United States"]
  region_res = call_gemini_lite(gemini_client,
                                        "Extract the region of the event\n"
                                        "Only return the region name\n"
                                        "Only pick from the following\n"
                                        "Return the exact Value\n"
                                        + str(possible_regions) + "\n",
                                        event_description)
    ## Validate Gemini's results
  event_details["region"] = validate_gemini_res(region_res, possible_regions, "region")
  event_details["description"] = event_description
   # Event series is chosen from standardized list
  possible_series: List[str] = ["Watson Distinguished Speaker Series",
                                "Watson Institute Research Seminar Series",
                                "War in Ukraine",
                                "Security Studies Seminar",
                                "Senior Fellows",
                                "Israel-Palestine Lecture Series",
                                "OP Jindal Distinguished Lectures",
                                "Book Adda",
                                "South Asia Seminar",
                                "Art History from the South",
                                "Other"]
  event_res = call_gemini(gemini_client,
                                        "Check to see if this event is part of the following series\n"
                                        + str(possible_series) +
                                        "Only return the series name\n"
                                        "it needs to be the exact series from above explicitly.\n"
                                        "If the event doesn't fit any of the above series, return Other\n",
                                        event_description)

      ## Validate Gemini's results
  event_details["event_series"] = validate_gemini_res(event_res, possible_series, "series")



      # General Category is chosen by gemini from the description
  possible_categories = ["Governance & Democracy",
                         "Conflict, Security & Foreign Policy",
                         "Global Economics & Development",
                         "Law, Rights & Justice",
                         "Migration & Diaspora Studies",
                         "Climate, Energy & Environment",
                         "Health, Medicine & Public Policy",
                         "Technology, Media & Society",
                         "Race, Gender & Social Inequality",
                         "History, Culture & Humanities",
                         "Education, Labor & Social Welfare",
                         "Urban Studies & Infrastructure",
                         "Research Methods & Academic Conversations"]
  category_res = call_gemini(gemini_client,
                                                "Choose from the following categories\n"
                                                + str(possible_categories) +
                                                "Only return the chosen name\n"
                                                "Given the following descriptions\n"
                                                "1. Governance & Democracy: Events focused on political systems, electoral processes, institutional design, state capacity, and democratic or authoritarian governance. Includes elections, constitutions, political parties, public administration, and leadership transitions.\n"
                                                "2. Conflict, Security & Foreign Policy: Covers domestic or international conflict, peacebuilding, military policy, national security, intelligence, diplomacy, humanitarian interventions, geopolitics, and state-to-state strategic relations.\n"
                                                "3. Global Economics & Development: Work on economic systems, inequality, development policy, markets, trade, fiscal/monetary politics, post-colonial development structures, and global capitalism. Includes modernization, macroeconomics, finance, and industrial transformation.\n"
                                                "4. Law, Rights & Justice: Topics involving legal systems, human rights, criminal justice, courts, incarceration, civil liberties, policing, legal reform, constitutional interpretation, or any event framed around rights and justice structures.\n"
                                                "5. Migration & Diaspora Studies: Focuses on human movement, refugee crises, borders, displacement, asylum policy, integration, transnational identities, diasporic studies, and global patterns of mobility.\n"
                                                "6. Climate, Energy & Environment: Includes climate science, sustainability, environmental justice, carbon policy, energy transitions, environmental disasters, resilience planning, and global climate governance.\n"
                                                "7. Health, Medicine & Public Policy: Events centered on healthcare systems, pandemics, medical ethics, epidemiology, social determinants of health, global health infrastructure, and public health governance.\n"
                                                "8. Technology, Media & Society: Covers journalism, press freedom, digital infrastructure, AI, misinformation, cybersecurity, surveillance, platform governance, and the societal implications of technological transformation.\n"
                                                "9. Race, Gender & Social Inequality: Explores inequity across identity categories, including race, ethnicity, caste, gender, sexuality, LGBTQ+ studies, civil rights, discrimination, social movements, equity policy, and intersectional research.\n"
                                                "10. History, Culture & Humanities: Events grounded in historical analysis, political history, cultural identity, literature, religion, ethics, art, narrative, or intellectual traditions, especially where these lenses are used to interpret contemporary or geopolitical realities.\n"
                                                "11. Education, Labor & Social Welfare: Examines labor markets, workers rights, unions, welfare policy, social safety nets, education equity, pedagogical policy, universities, and social mobility through institutional structures.\n"
                                                "12. Urban Studies & Infrastructure: Focus on cities, urban policy, transportation systems, infrastructure planning, housing, gentrification, inequality shaped by urban design, and governance of metropolitan spaces.\n"
                                                "13. Research Methods & Academic Conversations: Covers book talks, workshops, panels, methodology discussions, disciplinary theory, working papers, scholarly debates, and any event primarily centered on academic research practice rather than a specific policy topic.\n",
                                                event_description)
    ## Validate Gemini Results
  event_details["general_category"] = validate_gemini_res(category_res, possible_categories, "general_category")

  # Student Run
  event_details["student_run"] = "N"
  return event_details

if __name__ == "__main__":
  driver = gs.Chrome()

  while True:
    url = input("Enter the URL of the event from events@brown: ")
    if url.lower() == "quit" or url.lower() == "exit":
      print("Exiting....")
      break
    expected_pattern = r'^https://events\.brown\.edu(/.*)?$'
    if not re.match(expected_pattern, url):
      print("Invalid URL. Please enter a valid URL.")
      continue
    event_details = extract_event_details(url)
    df = pd.DataFrame(event_details, index=[0])

    # Display as tab-separated for pasting into Sheets
    data_table.enable_dataframe_formatter()
    # tsv_output = df.to_csv(sep='\t', index=False)
    tsv_output = df.to_csv(sep='\t', index=False, header=False)
    display(HTML(f"<textarea style='width:100%; height:100px;'>{tsv_output}</textarea>"))
  driver.quit()

    # Example urls for testing
    # "https://events.brown.edu/event/313473-thea-riofrancos-extraction-the-frontiers-of-green-cap"
    # "https://events.brown.edu/event/321735-understanding-the-government-shutdown-causes-and"
    # "https://events.brown.edu/event/immigrationjournalism"
    # "https://events.brown.edu/event/291707-indian-elections-and-after"
    # "https://events.brown.edu/event/293910-climate-action-and-the-2024-election-scienc".


<IPython.core.display.Javascript object>

Enter the URL of the event from events@brown: https://events.brown.edu/event/313473-thea-riofrancos-extraction-the-frontiers-of-green-cap
Opening URL...
Extracting event details...


Enter the URL of the event from events@brown: quit
Exiting....


Wait for above cell 🔼 to be ready. You should see an input box soon.
Type "exit" in it to terminate the program.

It will do the following steps during setup:
![Screenshot of Setup steps](https://i.imgur.com/Xto4WAn.png)